In [1]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
 """))

In [24]:
# 검색할 텍스트 가져오기 
with open('0709_데이터 폴더-카피본/search_text(0710_15)/search_keyword.txt', 'r', encoding='utf-8') as f:
    search_text = [line.strip() for line in f if line.strip()]
    
print(search_text)

['노브랜드버거 NBB 어메이징 더블 버거세트,노브랜드버거 닭가슴살앤두부 샐러드 빅 콤보,노브랜드버거 바질 에그마요 버거세트,노브랜드버거 여기어때 클럽 샌드위치 버거,노브랜드버거 여기어때 클럽 샌드위치 버거 패키지,노브랜드버거 치킨 시저 샐러드 콤보,노브랜드버거 통마늘 베이컨 버거세트,노브랜드버거 허브순살치킨런(M),롯데리아 김치불고기버거,롯데리아 김치불고기버거 세트,롯데리아 더블 데리버거 세트,롯데리아 더블 미라클버거 세트,롯데리아 더블 치킨버거 세트,롯데리아 더블 클래식치즈버거 세트,롯데리아 더블엑스투버거 세트,롯데리아 데리버거 세트,롯데리아 리아 불고기 베이컨 세트,롯데리아 리아 불고기 세트,롯데리아 리아 사각새우 더블 세트,롯데리아 리아 새우 베이컨 세트,롯데리아 리아 새우 세트,롯데리아 모짜렐라버거 토마토바질,롯데리아 모짜렐라버거세트 발사믹바질,롯데리아 모짜렐라버거세트 토마토바질,롯데리아 못난이치즈감자,롯데리아 미라클버거 세트,롯데리아 에그김치불고기버거,롯데리아 에그김치불고기버거 세트,롯데리아 오징어 얼라이브버거 매운맛,롯데리아 오징어 얼라이브버거 매운맛 세트,롯데리아 오징어 얼라이브버거 블랙페퍼맛,롯데리아 오징어 얼라이브버거 블랙페퍼맛 세트,롯데리아 전주비빔라이스버거세트,롯데리아 치킨버거 세트,롯데리아 클래식치즈버거 세트,롯데리아 티렉스버거 세트,롯데리아 한우불고기버거 세트,롯데리아 핫크리스피치킨버거 세트,맥도날드 1955 버거™ 세트,맥도날드 맥스파이시® 상하이 버거 세트,맥도날드 빅맥® 세트,버거리 비프앤쉬림프버거,버거리 소불고기버거,버거리 쉬림프버거,버거리 에그불고기버거,버거리 프레쉬버거,버거운버거 (NEW)닭가슴살 샐러드,버거운버거 (NEW)베이컨에그치즈베이크 세트(세트),버거운버거 (NEW)쉬림프샐러드,버거운버거 (NEW)에그베이컨샐러드,버거운버거 (NEW)치킨샐러드,버거운버거 (NEW)휠렛불갈비치즈베이크 세트(세트),버거운버거 달콤비프치즈베이크(단품),버거운버거 매콤치킨치즈베이크 세트(세트),버거운버거 매콤치킨치즈베이크(단품),버거운버거 

In [9]:
# 환경 변수 체크 
from  dotenv import load_dotenv
import os

load_dotenv()

True

In [22]:
# 네이버 이미지 검색 
import requests
import json
import time 
import pandas as pd
from hashlib import md5

# 네이버 API 정보 
client_id = os.getenv('Client_ID')
client_secret = os.getenv('Client_Secret')

#이미지 저장 기준
base_dir = '0709_데이터 폴더-카피본'
max_images_per_menu = 10
exclude_keywords = [""] # 검색어 제외할 키워드

# 1. 검색어 리스트 불러오기(쉼표로 나눠진 브랜드명+메뉴명)
with open('0709_데이터 폴더-카피본/search_text(0710)/search_keyword.txt','r',encoding='utf-8') as f:
    lines = f.readlines()
    search_keyword = []
    for line in lines:
        items = [x.strip() for x in line.strip().split(',') if x.strip()]
        search_keyword.extend(items)

# 2. 네이버 이미지 검색 함수 (title 함께 return)
def search_img_naver(query, display=1, start=1): # 여기 display 나중에 100으로 바꾸기
    url = "https://openapi.naver.com/v1/search/image"
    headers={
        "X-Naver-Client-Id":client_id,
        "X-Naver-Client-Secret":client_secret
    }
    params = {
        "query":query,
        "display":display,
        "start":start,
        "sort":"sim",      # 유사도 기반
        "filter":"large"   # 큰 이미지만 저장
    }
    response = requests.get(url,headers=headers, params=params)
    if response.status_code == 200:
        items = response.json().get('item', [])
        return[(item['link'], item['title']) for item in itmes]
    else:
        print(f"[{query}] 요청 실패 : {response.status_code}")
        return []
    
# 3. 네이버 이미지 저장 함수
def save_imgae(url, save_path, count):
    try:
        ext = url.split('.')[-1].split('?')[0].lower()
        if ext not in ['jpg', 'jpeg', 'png', 'gif', 'webp']:
            ext = 'jpg'
        filename = f"review_img__{count:04d}.{ext}"
        filepath = os.path.join(save_path, filename)
        urlretrieve(url, filepath)
        return True
    except Exception as e:
        print(f"→ 저장 실패: {e}")
        return False
    
    
#=============================================전체 실행 코드====================================
for keyword in search_keyword:
    try:
        brand,menu = keyword.split(' ', 1)
    except ValueError:
        print(f"❌ 잘못된 포맷: {keyword}")
        continue
        
    target_path = os.path.join(base_dir, brand, menu)
    if not os.path.isdir(target_path):
        print(f"❌ 폴더 없음 : {target_path}")
        continue
    print(f"\n ✅ 검색어 : {keyword} → 저장 폴더 : {target_path}")
    
    saved_count = 0
    start = 1
    while saved_count < target_path_count :
        image_items = search_img_naver(keyword, display=10, start=start)
        if not image_items:
            print('💦 더이상 결과 없음')
            break
            
        for link,tilte in image_items:
            if any(ex_kw in title.lower() for ex_kw in exclude_keywords_in_title):
                continue # 제목 필터 제외
                
            if save_imgae(link, target_path, saved_count +1):
                save_imgae += 1
                print(f"💥저장 성공 ({saved_count}/{target_image_count})")
            if saved_count >= target_image_count:
                break
                
            time.sleep(0.2)
        start =+ 100
    print(f"📢[{keyword}]최종 저장 완료 : {saved_count}장\n")

['노브랜드버거 NBB 어메이징 더블 버거세트', '노브랜드버거 닭가슴살앤두부 샐러드 빅 콤보', '노브랜드버거 바질 에그마요 버거세트', '노브랜드버거 여기어때 클럽 샌드위치 버거', '노브랜드버거 여기어때 클럽 샌드위치 버거 패키지', '노브랜드버거 치킨 시저 샐러드 콤보', '노브랜드버거 크런치 새우볼(3조각)', '노브랜드버거 통마늘 베이컨 버거세트', '노브랜드버거 허브순살치킨런(M)', '롯데리아 김치불고기버거', '롯데리아 김치불고기버거 세트', '롯데리아 더블 데리버거 세트', '롯데리아 더블 미라클버거 세트', '롯데리아 더블 치킨버거 세트', '롯데리아 더블 클래식치즈버거 세트', '롯데리아 더블엑스투버거 세트', '롯데리아 데리버거 세트', '롯데리아 리아 불고기 베이컨 세트', '롯데리아 리아 불고기 세트', '롯데리아 리아 사각새우 더블 세트', '롯데리아 리아 새우 베이컨 세트', '롯데리아 리아 새우 세트', '롯데리아 모짜렐라버거 토마토바질', '롯데리아 모짜렐라버거세트 발사믹바질', '롯데리아 모짜렐라버거세트 토마토바질', '롯데리아 못난이치즈감자', '롯데리아 미라클버거 세트', '롯데리아 에그김치불고기버거', '롯데리아 에그김치불고기버거 세트', '롯데리아 오징어 얼라이브버거 매운맛', '롯데리아 오징어 얼라이브버거 매운맛 세트', '롯데리아 오징어 얼라이브버거 블랙페퍼맛', '롯데리아 오징어 얼라이브버거 블랙페퍼맛 세트', '롯데리아 전주비빔라이스버거세트', '롯데리아 쥐포튀김(청양마요소스)', '롯데리아 지파이 고소한맛(S)', '롯데리아 지파이 하바네로(L)', '롯데리아 치킨버거 세트', '롯데리아 치킨휠레 4조각', '롯데리아 클래식치즈버거 세트', '롯데리아 티렉스버거 세트', '롯데리아 한우불고기버거 세트', '롯데리아 핫크리스피치킨버거 세트', '롯데리아 화이어윙 2조각', '롯데리아 화이어윙 4조각', '맥도날드 1955 버거™ 세트', '맥도날드 맥스파이시® 상하이 버거 세트', '맥도날드 빅맥® 세트', '

In [1]:
# 네이버 이미지 검색 (클로드 버전)
import requests
import json
import time 
import pandas as pd
import os
from hashlib import md5
from urllib.request import urlretrieve

# 네이버 API 정보 
client_id = os.getenv('Client_ID')
client_secret = os.getenv('Client_Secret')

# 이미지 저장 기준
base_dir = '0709_데이터 폴더-카피본'
max_images_per_menu = 5  # 메뉴당 최대 이미지 수
exclude_keywords_in_title = ["쇼핑몰", "츄리링", "바지", "로고", "아이콘", "텍스트"]  # 제목에서 제외할 키워드

print("="*60)
print("네이버 이미지 크롤링 시작")
print("="*60)

# 1. 검색어 리스트 불러오기
try:
    with open('0709_데이터 폴더-카피본/search_text(0710)/search_keyword.txt', 'r', encoding='utf-8') as f:
        lines = f.readlines()
        search_keywords = []
        for line in lines:
            # 각 줄에서 공백 제거하고 빈 줄 제외
            keyword = line.strip()
            if keyword:  # 빈 줄이 아닌 경우만 추가
                search_keywords.append(keyword)
                
    print(f"📋 총 {len(search_keywords)}개의 검색어를 불러왔습니다.")
    print("첫 5개 검색어:", search_keywords[:5])
    
except FileNotFoundError:
    print("❌ 검색어 파일을 찾을 수 없습니다.")
    exit()

# 2. 네이버 이미지 검색 함수 (title 함께 return)
def search_img_naver(query, display=100, start=1):
    """네이버 이미지 검색 API 호출"""
    url = "https://openapi.naver.com/v1/search/image"
    headers = {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret
    }
    params = {
        "query": query,
        "display": display,  # 한 번에 가져올 이미지 수 (최대 100)
        "start": start,
        "sort": "sim",       # 유사도 기반
        "filter": "large"    # 큰 이미지만 저장
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            items = response.json().get('items', [])  # 'item' → 'items'로 수정
            return [(item['link'], item['title']) for item in items]
        else:
            print(f"[{query}] 요청 실패: {response.status_code}")
            return []
    except Exception as e:
        print(f"[{query}] API 호출 에러: {e}")
        return []

# 3. 네이버 이미지 저장 함수
def save_image(url, save_path, count):
    """이미지 다운로드 및 저장"""
    try:
        # 확장자 추출
        ext = url.split('.')[-1].split('?')[0].lower()
        if ext not in ['jpg', 'jpeg', 'png', 'gif', 'webp']:
            ext = 'jpg'
        
        # 파일명 생성: naver_img_0001.jpg 형태
        filename = f"naver_img_{count:04d}.{ext}"
        filepath = os.path.join(save_path, filename)
        
        # 이미지 다운로드
        urlretrieve(url, filepath)
        return True
        
    except Exception as e:
        print(f"→ 저장 실패: {e}")
        return False

# 4. 안전한 폴더명 생성 함수
def safe_folder_name(name):
    """파일 시스템에 안전한 폴더명 생성"""
    unsafe_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*']
    safe_name = name
    for char in unsafe_chars:
        safe_name = safe_name.replace(char, '_')
    return safe_name.strip()

# 5. 폴더 생성 함수
def create_folder_if_not_exists(folder_path):
    """폴더가 없으면 생성"""
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"📁 폴더 생성: {folder_path}")
        return True
    return False

# =============================================전체 실행 코드====================================
print(f"\n🚀 이미지 크롤링 시작 - 메뉴당 최대 {max_images_per_menu}장")
print("="*60)

total_success = 0
total_failed = 0

for idx, keyword in enumerate(search_keywords, 1):
    print(f"\n[{idx}/{len(search_keywords)}] 🔍 검색어: '{keyword}'")
    
    try:
        # 브랜드명과 메뉴명 분리
        parts = keyword.split(' ', 1)
        if len(parts) < 2:
            print(f"❌ 잘못된 포맷 (공백으로 구분되지 않음): {keyword}")
            total_failed += 1
            continue
            
        brand, menu = parts[0], parts[1]
        
        # 안전한 폴더명으로 변환
        safe_brand = safe_folder_name(brand)
        safe_menu = safe_folder_name(menu)
        
        # 저장 경로 설정
        target_path = os.path.join(base_dir, safe_brand, safe_menu)
        
        # 폴더 생성
        create_folder_if_not_exists(target_path)
        
        print(f"📂 저장 폴더: {target_path}")
        
        # 이미지 검색 및 저장
        saved_count = 0
        start = 1
        
        while saved_count < max_images_per_menu:
            # 한 번에 최대 100개씩 검색
            display_count = min(100, max_images_per_menu - saved_count)
            image_items = search_img_naver(keyword, display=display_count, start=start)
            
            if not image_items:
                print('💧 더 이상 검색 결과 없음')
                break
            
            print(f"🔎 {len(image_items)}개 이미지 발견, 저장 중...")
            
            for link, title in image_items:
                # 제외할 키워드가 제목에 포함되어 있으면 건너뛰기
                if any(ex_kw in title.lower() for ex_kw in exclude_keywords_in_title):
                    print(f"→ 제외됨 (키워드 필터): {title[:50]}...")
                    continue
                
                # 이미지 저장 시도
                if save_image(link, target_path, saved_count + 1):
                    saved_count += 1
                    print(f"✅ 저장 성공 ({saved_count}/{max_images_per_menu}): naver_img_{saved_count:04d}")
                else:
                    print(f"❌ 저장 실패: {link[:50]}...")
                
                # 목표 개수에 도달하면 중단
                if saved_count >= max_images_per_menu:
                    break
                
                # API 요청 제한을 위한 딜레이
                time.sleep(0.1)
            
            # 다음 페이지로
            start += 100
            
            # 한 번에 검색 결과가 100개 미만이면 더 이상 없다고 판단
            if len(image_items) < 100:
                break
        
        print(f"📊 [{keyword}] 최종 저장 완료: {saved_count}장")
        total_success += saved_count
        
        # 검색어 간 딜레이 (API 제한 고려)
        time.sleep(1)
        
    except Exception as e:
        print(f"❌ [{keyword}] 처리 중 오류 발생: {e}")
        total_failed += 1
        continue

print("\n" + "="*60)
print("🎉 크롤링 완료!")
print(f"📊 총 저장된 이미지: {total_success}장")
print(f"❌ 실패한 검색어: {total_failed}개")
print("="*60)

# 결과 로그 저장
log_filename = os.path.join(base_dir, 'crawling_log.txt')
with open(log_filename, 'w', encoding='utf-8') as f:
    f.write(f"네이버 이미지 크롤링 결과\n")
    f.write(f"처리 시간: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"총 검색어 수: {len(search_keywords)}\n")
    f.write(f"총 저장 이미지: {total_success}장\n")
    f.write(f"실패한 검색어: {total_failed}개\n")
    f.write("="*50 + "\n")

print(f"📝 로그 파일 저장: {log_filename}")

네이버 이미지 크롤링 시작
📋 총 1개의 검색어를 불러왔습니다.
첫 5개 검색어: ['노브랜드버거 NBB 어메이징 더블 버거세트,노브랜드버거 닭가슴살앤두부 샐러드 빅 콤보,노브랜드버거 바질 에그마요 버거세트,노브랜드버거 여기어때 클럽 샌드위치 버거,노브랜드버거 여기어때 클럽 샌드위치 버거 패키지,노브랜드버거 치킨 시저 샐러드 콤보,노브랜드버거 크런치 새우볼(3조각),노브랜드버거 통마늘 베이컨 버거세트,노브랜드버거 허브순살치킨런(M),롯데리아 김치불고기버거,롯데리아 김치불고기버거 세트,롯데리아 더블 데리버거 세트,롯데리아 더블 미라클버거 세트,롯데리아 더블 치킨버거 세트,롯데리아 더블 클래식치즈버거 세트,롯데리아 더블엑스투버거 세트,롯데리아 데리버거 세트,롯데리아 리아 불고기 베이컨 세트,롯데리아 리아 불고기 세트,롯데리아 리아 사각새우 더블 세트,롯데리아 리아 새우 베이컨 세트,롯데리아 리아 새우 세트,롯데리아 모짜렐라버거 토마토바질,롯데리아 모짜렐라버거세트 발사믹바질,롯데리아 모짜렐라버거세트 토마토바질,롯데리아 못난이치즈감자,롯데리아 미라클버거 세트,롯데리아 에그김치불고기버거,롯데리아 에그김치불고기버거 세트,롯데리아 오징어 얼라이브버거 매운맛,롯데리아 오징어 얼라이브버거 매운맛 세트,롯데리아 오징어 얼라이브버거 블랙페퍼맛,롯데리아 오징어 얼라이브버거 블랙페퍼맛 세트,롯데리아 전주비빔라이스버거세트,롯데리아 쥐포튀김(청양마요소스),롯데리아 지파이 고소한맛(S),롯데리아 지파이 하바네로(L),롯데리아 치킨버거 세트,롯데리아 치킨휠레 4조각,롯데리아 클래식치즈버거 세트,롯데리아 티렉스버거 세트,롯데리아 한우불고기버거 세트,롯데리아 핫크리스피치킨버거 세트,롯데리아 화이어윙 2조각,롯데리아 화이어윙 4조각,맥도날드 1955 버거™ 세트,맥도날드 맥스파이시® 상하이 버거 세트,맥도날드 빅맥® 세트,버거리 비프앤쉬림프버거,버거리 소불고기버거,버거리 쉬림프버거,버거리 에그불고기버거,버거리 프레쉬버거,버거운버거 (NEW)닭가슴살 샐러드,버거운버거 (NEW)베이컨에그치즈베이크 

In [ ]:
# 다시 json으로 클롤링

In [ ]:
# 네이버 이미지 검색 
import requests
import json
import time 
import pandas as pd
import os
from hashlib import md5
from urllib.request import urlretrieve
from dotenv import load_dotenv
load_dotenv()


# 네이버 API 정보 
client_id = os.getenv('Client_ID')
client_secret = os.getenv('Client_Secret')
print(client_id,client_secret)

# 이미지 저장 기준
base_dir = '0709_데이터 폴더-카피본'
max_images_per_menu = 300  # 메뉴당 최대 이미지 수
exclude_keywords_in_title = ["쇼핑몰", "츄리링", "바지","의상","하의","상의"]  # 제목에서 제외할 키워드

print("="*60)
print("네이버 이미지 크롤링 시작")
print("="*60)

# 1. JSON 파일에서 검색어 리스트 불러오기
try:
    # JSON 파일 경로 - 실제 파일 경로로 수정하세요
    json_file_path = '0709_데이터 폴더-카피본/search_text(0710_15)/brand_menu.json'
    
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # JSON 구조에 따라 검색어 추출
    search_keywords = []
    
    # 방법 1: 단순 리스트 형태인 경우
    if isinstance(data, list):
        search_keywords = data
    
    # 방법 2: 딕셔너리 형태인 경우 (노브랜드버거 같은 키가 있는 경우)
    elif isinstance(data, dict):
        for brand_name, menu_list in data.items():
            if isinstance(menu_list, list):
                # 브랜드명 + 메뉴명 형태로 조합
                for menu in menu_list:
                    search_keywords.append(f"{brand_name} {menu}")
            else:
                # 단일 값인 경우
                search_keywords.append(f"{brand_name} {menu_list}")
    
    print(f"📋 총 {len(search_keywords)}개의 검색어를 불러왔습니다.")
#     print("첫 5개 검색어:")
#     for i, keyword in enumerate(search_keywords[:5], 1):
#         print(f"  {i}. {keyword}")
    
#     if len(search_keywords) > 5:
#         print(f"  ... 외 {len(search_keywords) - 5}개 더")
    
#     JSON 구조 확인을 위한 디버그 정보
    print(f"\n🔍 JSON 파일 구조 확인:")
    print(f"   - 데이터 타입: {type(data)}")
    if isinstance(data, dict):
        print(f"   - 키 개수: {len(data)}")
        print(f"   - 첫 번째 키: {list(data.keys())[0] if data else 'None'}")
    
except FileNotFoundError:
    print("❌ JSON 파일을 찾을 수 없습니다.")
    print(f"파일 경로를 확인하세요: {json_file_path}")
    exit()
except json.JSONDecodeError as e:
    print(f"❌ JSON 파일 형식 오류: {e}")
    exit()
except Exception as e:
    print(f"❌ JSON 파일 읽기 오류: {e}")
    exit()

# 2. 네이버 이미지 검색 함수 (title 함께 return)
def search_img_naver(query, display=100, start=1):
    """네이버 이미지 검색 API 호출"""
    url = "https://openapi.naver.com/v1/search/image"
    headers = {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret
    }
    params = {
        "query": query,
        "display": display,  # 한 번에 가져올 이미지 수 (최대 100)
        "start": start,
        "sort": "sim",       # 유사도 기반
        "filter": "large"    # 큰 이미지만 저장
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            items = response.json().get('items', [])  # 'item' → 'items'로 수정
            return [(item['link'], item['title']) for item in items]
        else:
            print(f"[{query}] 요청 실패: {response.status_code}")
            return []
    except Exception as e:
        print(f"[{query}] API 호출 에러: {e}")
        return []

# 3. 네이버 이미지 저장 함수
def save_image(url, save_path, count):
    """이미지 다운로드 및 저장"""
    try:
        # 확장자 추출
        ext = url.split('.')[-1].split('?')[0].lower()
        if ext not in ['jpg', 'jpeg', 'png', 'gif', 'webp']:
            ext = 'jpg'
        
        # 파일명 생성: naver_img_0001.jpg 형태
        filename = f"naver_img_{count:04d}.{ext}"
        filepath = os.path.join(save_path, filename)
        
        # 이미지 다운로드
        urlretrieve(url, filepath)
        return True
        
    except Exception as e:
        print(f"→ 저장 실패: {e}")
        return False

# 4. 안전한 폴더명 생성 함수
def safe_folder_name(name):
    """파일 시스템에 안전한 폴더명 생성"""
    unsafe_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*']
    safe_name = name
    for char in unsafe_chars:
        safe_name = safe_name.replace(char, '_')
    return safe_name.strip()

# 5. 폴더 생성 함수
def create_folder_if_not_exists(folder_path):
    """폴더가 없으면 생성"""
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"📁 폴더 생성: {folder_path}")
        return True
    return False

# =============================================전체 실행 코드====================================
print(f"\n🚀 이미지 크롤링 시작 - 메뉴당 최대 {max_images_per_menu}장")
print("="*60)

total_success = 0
total_failed = 0

for idx, keyword in enumerate(search_keywords, 1):
    print(f"\n[{idx}/{len(search_keywords)}] 🔍 검색어: '{keyword}'")
    
    try:
        # 브랜드명과 메뉴명 분리
        parts = keyword.split(' ', 1)
        if len(parts) < 2:
            print(f"❌ 잘못된 포맷 (공백으로 구분되지 않음): {keyword}")
            total_failed += 1
            continue
            
        brand, menu = parts[0], parts[1]
        
        # 안전한 폴더명으로 변환
        safe_brand = safe_folder_name(brand)
        safe_menu = safe_folder_name(menu)
        
        # 저장 경로 설정
        target_path = os.path.join(base_dir, safe_brand, safe_menu)
        
        # 폴더 생성
        create_folder_if_not_exists(target_path)
        
        print(f"📂 저장 폴더: {target_path}")
        
        # 이미지 검색 및 저장
        saved_count = 0
        start = 1
        
        while saved_count < max_images_per_menu:
            # 한 번에 최대 100개씩 검색
            display_count = min(100, max_images_per_menu - saved_count)
            image_items = search_img_naver(keyword, display=display_count, start=start)
            
            if not image_items:
                print('💧 더 이상 검색 결과 없음')
                break
            
            print(f"🔎 {len(image_items)}개 이미지 발견, 저장 중...")
            
            for link, title in image_items:
                # 제외할 키워드가 제목에 포함되어 있으면 건너뛰기
                if any(ex_kw in title.lower() for ex_kw in exclude_keywords_in_title):
                    print(f"→ 제외됨 (키워드 필터): {title[:50]}...")
                    continue
                
                # 이미지 저장 시도
                if save_image(link, target_path, saved_count + 1):
                    saved_count += 1
                    print(f"✅ 저장 성공 ({saved_count}/{max_images_per_menu}): naver_img_{saved_count:04d}")
                else:
                    print(f"❌ 저장 실패: {link[:50]}...")
                
                # 목표 개수에 도달하면 중단
                if saved_count >= max_images_per_menu:
                    break
                
                # API 요청 제한을 위한 딜레이
                time.sleep(0.1)
            
            # 다음 페이지로
            start += 100
            
            # 한 번에 검색 결과가 100개 미만이면 더 이상 없다고 판단
            if len(image_items) < 100:
                break
        
        print(f"📊 [{keyword}] 최종 저장 완료: {saved_count}장")
        total_success += saved_count
        
        # 검색어 간 딜레이 (API 제한 고려)
        time.sleep(1)
        
    except Exception as e:
        print(f"❌ [{keyword}] 처리 중 오류 발생: {e}")
        total_failed += 1
        continue

print("\n" + "="*60)
print("🎉 크롤링 완료!")
print(f"📊 총 저장된 이미지: {total_success}장")
print(f"❌ 실패한 검색어: {total_failed}개")
print("="*60)

# 결과 로그 저장
log_filename = os.path.join(base_dir, 'crawling_log.txt')
with open(log_filename, 'w', encoding='utf-8') as f:
    f.write(f"네이버 이미지 크롤링 결과\n")
    f.write(f"처리 시간: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"총 검색어 수: {len(search_keywords)}\n")
    f.write(f"총 저장 이미지: {total_success}장\n")
    f.write(f"실패한 검색어: {total_failed}개\n")
    f.write("="*50 + "\n")

print(f"📝 로그 파일 저장: {log_filename}")

SxIc_HSA3r4D_7dvoq1p gJ67RkhzTA
네이버 이미지 크롤링 시작
📋 총 141개의 검색어를 불러왔습니다.

🔍 JSON 파일 구조 확인:
   - 데이터 타입: <class 'dict'>
   - 키 개수: 9
   - 첫 번째 키: 노브랜드버거

🚀 이미지 크롤링 시작 - 메뉴당 최대 300장

[1/141] 🔍 검색어: '노브랜드버거 NBB 어메이징 더블 버거세트'
📂 저장 폴더: 0709_데이터 폴더-카피본\노브랜드버거\NBB 어메이징 더블 버거세트
🔎 100개 이미지 발견, 저장 중...
✅ 저장 성공 (1/300): naver_img_0001
✅ 저장 성공 (2/300): naver_img_0002
✅ 저장 성공 (3/300): naver_img_0003
✅ 저장 성공 (4/300): naver_img_0004
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2023/10/03/316f...
✅ 저장 성공 (5/300): naver_img_0005
✅ 저장 성공 (6/300): naver_img_0006
✅ 저장 성공 (7/300): naver_img_0007
✅ 저장 성공 (8/300): naver_img_0008
✅ 저장 성공 (9/300): naver_img_0009
✅ 저장 성공 (10/300): naver_img_0010
✅ 저장 성공 (11/300): naver_img_0011
✅ 저장 성공 (12/300): naver_img_0012
→ 저장 실패: <urlopen error [Errno 11001] getaddrinfo failed>
❌ 저장 실패: https://cdn.clien.net/web/api/file/F01/9525139/b13...
✅ 저장 성공 (13/300): naver_img_0013
✅ 저장 성공 (14/300): naver_img_0014
✅ 저장 성공 (15/300): naver_img_0015
✅ 저장

✅ 저장 성공 (139/300): naver_img_0139
✅ 저장 성공 (140/300): naver_img_0140
✅ 저장 성공 (141/300): naver_img_0141
✅ 저장 성공 (142/300): naver_img_0142
✅ 저장 성공 (143/300): naver_img_0143
✅ 저장 성공 (144/300): naver_img_0144
✅ 저장 성공 (145/300): naver_img_0145
✅ 저장 성공 (146/300): naver_img_0146
✅ 저장 성공 (147/300): naver_img_0147
✅ 저장 성공 (148/300): naver_img_0148
✅ 저장 성공 (149/300): naver_img_0149
✅ 저장 성공 (150/300): naver_img_0150
✅ 저장 성공 (151/300): naver_img_0151
✅ 저장 성공 (152/300): naver_img_0152
✅ 저장 성공 (153/300): naver_img_0153
✅ 저장 성공 (154/300): naver_img_0154
✅ 저장 성공 (155/300): naver_img_0155
✅ 저장 성공 (156/300): naver_img_0156
✅ 저장 성공 (157/300): naver_img_0157
✅ 저장 성공 (158/300): naver_img_0158
✅ 저장 성공 (159/300): naver_img_0159
✅ 저장 성공 (160/300): naver_img_0160
✅ 저장 성공 (161/300): naver_img_0161
✅ 저장 성공 (162/300): naver_img_0162
✅ 저장 성공 (163/300): naver_img_0163
✅ 저장 성공 (164/300): naver_img_0164
✅ 저장 성공 (165/300): naver_img_0165
✅ 저장 성공 (166/300): naver_img_0166
✅ 저장 성공 (167/300): naver_img_0167
✅ 저장 성공 (168/3

✅ 저장 성공 (1/300): naver_img_0001
✅ 저장 성공 (2/300): naver_img_0002
✅ 저장 성공 (3/300): naver_img_0003
✅ 저장 성공 (4/300): naver_img_0004
✅ 저장 성공 (5/300): naver_img_0005
✅ 저장 성공 (6/300): naver_img_0006
✅ 저장 성공 (7/300): naver_img_0007
✅ 저장 성공 (8/300): naver_img_0008
✅ 저장 성공 (9/300): naver_img_0009
✅ 저장 성공 (10/300): naver_img_0010
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://scontent-nrt1-2.cdninstagram.com/v/t51.293...
✅ 저장 성공 (11/300): naver_img_0011
✅ 저장 성공 (12/300): naver_img_0012
✅ 저장 성공 (13/300): naver_img_0013
✅ 저장 성공 (14/300): naver_img_0014
✅ 저장 성공 (15/300): naver_img_0015
✅ 저장 성공 (16/300): naver_img_0016
✅ 저장 성공 (17/300): naver_img_0017
✅ 저장 성공 (18/300): naver_img_0018
✅ 저장 성공 (19/300): naver_img_0019
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2023/11/07/639b...
✅ 저장 성공 (20/300): naver_img_0020
✅ 저장 성공 (21/300): naver_img_0021
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://scontent-nrt1-2.cdninstagram.com/v/t51.293...
✅ 저장 성공 (22/300): nave

✅ 저장 성공 (51/300): naver_img_0051
✅ 저장 성공 (52/300): naver_img_0052
✅ 저장 성공 (53/300): naver_img_0053
✅ 저장 성공 (54/300): naver_img_0054
✅ 저장 성공 (55/300): naver_img_0055
✅ 저장 성공 (56/300): naver_img_0056
✅ 저장 성공 (57/300): naver_img_0057
✅ 저장 성공 (58/300): naver_img_0058
✅ 저장 성공 (59/300): naver_img_0059
✅ 저장 성공 (60/300): naver_img_0060
✅ 저장 성공 (61/300): naver_img_0061
✅ 저장 성공 (62/300): naver_img_0062
✅ 저장 성공 (63/300): naver_img_0063
✅ 저장 성공 (64/300): naver_img_0064
✅ 저장 성공 (65/300): naver_img_0065
✅ 저장 성공 (66/300): naver_img_0066
✅ 저장 성공 (67/300): naver_img_0067
✅ 저장 성공 (68/300): naver_img_0068
✅ 저장 성공 (69/300): naver_img_0069
✅ 저장 성공 (70/300): naver_img_0070
✅ 저장 성공 (71/300): naver_img_0071
✅ 저장 성공 (72/300): naver_img_0072
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://images.pexels.com/photos/29935343/pexels-p...
✅ 저장 성공 (73/300): naver_img_0073
✅ 저장 성공 (74/300): naver_img_0074
✅ 저장 성공 (75/300): naver_img_0075
✅ 저장 성공 (76/300): naver_img_0076
✅ 저장 성공 (77/300): naver_img_0077
✅ 저장 성공 (78

✅ 저장 성공 (147/300): naver_img_0147
→ 저장 실패: HTTP Error 500: Internal Server Error
❌ 저장 실패: https://cdn.ppomppu.co.kr/zboard/data3/2021/0628/2...
✅ 저장 성공 (148/300): naver_img_0148
✅ 저장 성공 (149/300): naver_img_0149
✅ 저장 성공 (150/300): naver_img_0150
✅ 저장 성공 (151/300): naver_img_0151
✅ 저장 성공 (152/300): naver_img_0152
✅ 저장 성공 (153/300): naver_img_0153
✅ 저장 성공 (154/300): naver_img_0154
✅ 저장 성공 (155/300): naver_img_0155
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: http://cdn.dealbada.com/data/editor/1806/4a25d009c...
✅ 저장 성공 (156/300): naver_img_0156
✅ 저장 성공 (157/300): naver_img_0157
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2023/11/01/5972...
✅ 저장 성공 (158/300): naver_img_0158
✅ 저장 성공 (159/300): naver_img_0159
✅ 저장 성공 (160/300): naver_img_0160
→ 저장 실패: HTTP Error 500: Internal Server Error
❌ 저장 실패: https://cdn.ppomppu.co.kr/zboard/data3/2010/0613/1...
✅ 저장 성공 (161/300): naver_img_0161
✅ 저장 성공 (162/300): naver_img_0162
✅ 저장 성공 (163/300): naver_img_0163
✅ 저장 성

...

✅ 저장 성공 (3/300): naver_img_0003
→ 저장 실패: <urlopen error [Errno 11001] getaddrinfo failed>
❌ 저장 실패: https://cdn.clien.net/web/api/file/F01/10392636/47...
✅ 저장 성공 (4/300): naver_img_0004
✅ 저장 성공 (5/300): naver_img_0005
✅ 저장 성공 (6/300): naver_img_0006
✅ 저장 성공 (7/300): naver_img_0007
✅ 저장 성공 (8/300): naver_img_0008
✅ 저장 성공 (9/300): naver_img_0009
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2023/09/23/daa7...
✅ 저장 성공 (10/300): naver_img_0010
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: http://cdn.dealbada.com/restapi/image/resizeWidth/...
✅ 저장 성공 (11/300): naver_img_0011
✅ 저장 성공 (12/300): naver_img_0012
✅ 저장 성공 (13/300): naver_img_0013
✅ 저장 성공 (14/300): naver_img_0014
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2023/05/28/49d5...
✅ 저장 성공 (15/300): naver_img_0015
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: http://cdn.dealbada.com/restapi/image/resizeWidth/...
✅ 저장 성공 (16/300): naver_img_0016
→ 저장 실패: HTTP Error 403: Forbidd

✅ 저장 성공 (143/300): naver_img_0143
✅ 저장 성공 (144/300): naver_img_0144
✅ 저장 성공 (145/300): naver_img_0145
→ 저장 실패: HTTP Error 404: Not Found
❌ 저장 실패: http://befe.co.kr/upload/se/201607/EDITOR_20160720...
✅ 저장 성공 (146/300): naver_img_0146
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2023/06/25/5fd0...
→ 저장 실패: <urlopen error [Errno 11001] getaddrinfo failed>
❌ 저장 실패: https://bgm.gg/i/16b5595/resize...
✅ 저장 성공 (147/300): naver_img_0147
✅ 저장 성공 (148/300): naver_img_0148
→ 저장 실패: <urlopen error [WinError 10060] 연결된 구성원으로부터 응답이 없어 연결하지 못했거나, 호스트로부터 응답이 없어 연결이 끊어졌습니다>
❌ 저장 실패: http://image.coolenjoy.net/data/editor/1705/Bimg_2...
✅ 저장 성공 (149/300): naver_img_0149
✅ 저장 성공 (150/300): naver_img_0150
✅ 저장 성공 (151/300): naver_img_0151
→ 저장 실패: HTTP Error 404: Not Found
❌ 저장 실패: http://file3.instiz.net/data/cached_img/upload/202...
✅ 저장 성공 (152/300): naver_img_0152
✅ 저장 성공 (153/300): naver_img_0153
✅ 저장 성공 (154/300): naver_img_0154
→ 저장 실패: HTTP Error 403: Forbidden
❌

✅ 저장 성공 (272/300): naver_img_0272
✅ 저장 성공 (273/300): naver_img_0273
✅ 저장 성공 (274/300): naver_img_0274
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: http://www.dogdrip.net/dvs/b/i/18/04/19/37396208/3...
✅ 저장 성공 (275/300): naver_img_0275
✅ 저장 성공 (276/300): naver_img_0276
✅ 저장 성공 (277/300): naver_img_0277
✅ 저장 성공 (278/300): naver_img_0278
✅ 저장 성공 (279/300): naver_img_0279
✅ 저장 성공 (280/300): naver_img_0280
✅ 저장 성공 (281/300): naver_img_0281
✅ 저장 성공 (282/300): naver_img_0282
→ 저장 실패: <urlopen error [Errno 11001] getaddrinfo failed>
❌ 저장 실패: https://cdn.clien.net/web/api/file/F01/9943967/276...
✅ 저장 성공 (283/300): naver_img_0283
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2025/06/09/718c...
📊 [버거킹 통새우와퍼주니어] 최종 저장 완료: 283장

[111/141] 🔍 검색어: '버거킹 통새우와퍼주니어세트'
📂 저장 폴더: 0709_데이터 폴더-카피본\버거킹\통새우와퍼주니어세트
🔎 100개 이미지 발견, 저장 중...
✅ 저장 성공 (1/300): naver_img_0001
✅ 저장 성공 (2/300): naver_img_0002
✅ 저장 성공 (3/300): naver_img_0003
✅ 저장 성공 (4/300): naver_img_0004
→ 저장 실패: <urlopen

✅ 저장 성공 (111/300): naver_img_0111
→ 저장 실패: <urlopen error [Errno 11001] getaddrinfo failed>
❌ 저장 실패: https://bgm.gg/i/16b5595/resize...
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2023/06/25/5fd0...
✅ 저장 성공 (112/300): naver_img_0112
✅ 저장 성공 (113/300): naver_img_0113
✅ 저장 성공 (114/300): naver_img_0114
✅ 저장 성공 (115/300): naver_img_0115
✅ 저장 성공 (116/300): naver_img_0116
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: http://cdn.dealbada.com/restapi/image/resizeWidth/...
✅ 저장 성공 (117/300): naver_img_0117
✅ 저장 성공 (118/300): naver_img_0118
✅ 저장 성공 (119/300): naver_img_0119
✅ 저장 성공 (120/300): naver_img_0120
✅ 저장 성공 (121/300): naver_img_0121
✅ 저장 성공 (122/300): naver_img_0122
✅ 저장 성공 (123/300): naver_img_0123
✅ 저장 성공 (124/300): naver_img_0124
✅ 저장 성공 (125/300): naver_img_0125
✅ 저장 성공 (126/300): naver_img_0126
✅ 저장 성공 (127/300): naver_img_0127
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: https://img2.quasarzone.com/editor/2023/06/03/9d50...
✅ 저장 성공 (128/300): naver_i

✅ 저장 성공 (257/300): naver_img_0257
→ 저장 실패: <urlopen error [Errno 11001] getaddrinfo failed>
❌ 저장 실패: http://image.g9.co.kr/g/1571511566/n?ts=1551920715...
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: http://cdn.dealbada.com/data/editor/1702/3c205d530...
✅ 저장 성공 (258/300): naver_img_0258
✅ 저장 성공 (259/300): naver_img_0259
✅ 저장 성공 (260/300): naver_img_0260
✅ 저장 성공 (261/300): naver_img_0261
✅ 저장 성공 (262/300): naver_img_0262
✅ 저장 성공 (263/300): naver_img_0263
✅ 저장 성공 (264/300): naver_img_0264
✅ 저장 성공 (265/300): naver_img_0265
✅ 저장 성공 (266/300): naver_img_0266
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: http://cdn.dealbada.com/data/editor/1609/02383f7c7...
✅ 저장 성공 (267/300): naver_img_0267
✅ 저장 성공 (268/300): naver_img_0268
✅ 저장 성공 (269/300): naver_img_0269
✅ 저장 성공 (270/300): naver_img_0270
✅ 저장 성공 (271/300): naver_img_0271
✅ 저장 성공 (272/300): naver_img_0272
✅ 저장 성공 (273/300): naver_img_0273
✅ 저장 성공 (274/300): naver_img_0274
→ 저장 실패: HTTP Error 403: Forbidden
❌ 저장 실패: http://cdn.dealbada.com/res